## Model Development - Demo Version

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import os

### Update data_demo(no need to rerun)

In [6]:
# # Load both files
# data_demo = pd.read_excel('/Users/yanmiyu/Desktop/1470_final/DL-Marriage-Prediction/Data/data_demo.xlsx')
# update = pd.read_excel('/Users/yanmiyu/Desktop/1470_final/DL-Marriage-Prediction/Data/2013-2023_update.xlsx')

# # Select the 6 columns you want to add
# columns_to_add = ['url', 'filename', 'first_partner_level_id', 'second_partner_level_id', 'first_partner_field', 'second_partner_field']
# update_selected = update[columns_to_add]

# # Confirm both have the same number of rows
# assert len(data_demo) == len(update_selected), "Row counts do not match!"

# # Concatenate side-by-side
# combined = pd.concat([update_selected[['url', 'filename']], data_demo, update_selected[['first_partner_level_id', 'second_partner_level_id', 'first_partner_field', 'second_partner_field']]], axis=1)

# # Now reorder columns exactly as you want
# final_column_order = [
#     'url',
#     'filename',
#     'first_partner_gender',
#     'first_partner_age',
#     'second_partner_gender',
#     'second_partner_age',
#     'first_partner_school_category',
#     'second_partner_school_category',
#     'first_partner_level_id',
#     'second_partner_level_id',
#     'first_partner_field',
#     'second_partner_field'
# ]

# # Reorder
# combined = combined[final_column_order]

# # Save
# combined.to_excel('/Users/yanmiyu/Desktop/1470_final/DL-Marriage-Prediction/Data/data_demo.xlsx', index=False)


## 1. Load data & Preprocessing

In [22]:
# --- 1. Load Data ---
## df = pd.read_excel("../data/data_demo.xlsx")
df = pd.read_excel("../data/data_final.xlsx")

In [23]:
df.columns

Index(['url', 'filename', 'first_partner_gender', 'first_partner_age_bin',
       'second_partner_gender', 'second_partner_age_bin',
       'first_partner_school_category', 'second_partner_school_category',
       'first_partner_level_id', 'second_partner_level_id',
       'first_partner_field', 'second_partner_field'],
      dtype='object')

In [24]:
# --- 2. Encode categorical features ---

## Clean missing data
df['first_partner_age_bin'] = df['first_partner_age_bin'].replace('Not mentioned', 'Unknown')
df['second_partner_age_bin'] = df['second_partner_age_bin'].replace('Not mentioned', 'Unknown')

label_encoders = {}
columns_to_encode = [
    'first_partner_gender', 'first_partner_age_bin', 'first_partner_school_category',
    'second_partner_gender', 'second_partner_age_bin', 'second_partner_school_category',
    'first_partner_level_id', 'second_partner_level_id',
    'first_partner_field', 'second_partner_field'
]

for col in columns_to_encode:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))  # treat NaNs safely as "Unknown"
    label_encoders[col] = le

# --- 3. Prepare input and output tensors ---
input_cols = ['first_partner_gender', 'first_partner_age_bin', 'first_partner_school_category',
              'first_partner_level_id', 'first_partner_field']
output_cols = ['second_partner_gender', 'second_partner_age_bin', 'second_partner_school_category',
               'second_partner_level_id', 'second_partner_field']

# Split data
X_train, X_test, Y_train, Y_test = train_test_split(
    df[input_cols].values,
    df[output_cols].values,
    test_size=0.2,
    random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.long)
Y_train = torch.tensor(Y_train, dtype=torch.long)
Y_test = torch.tensor(Y_test, dtype=torch.long)

In [25]:
# --- 4. Improved TabTransformer-like model ---
class TabTransformer(nn.Module):
    def __init__(self, category_sizes, dim, output_sizes):
        super().__init__()
        # Embeddings for each categorical feature
        self.embeddings = nn.ModuleList([
            nn.Embedding(size, dim) for size in category_sizes
        ])

        # Transformer encoder
        self.transformer = nn.TransformerEncoder(
            encoder_layer=nn.TransformerEncoderLayer(
                d_model=dim,
                nhead=4,  # Increased attention heads
                dim_feedforward=128,
                dropout=0.1,
                batch_first=True  # Added to suppress warning
            ),
            num_layers=3,
            enable_nested_tensor=False  # Suppress warning
        )

        # Output heads with layer normalization
        self.heads = nn.ModuleList()
        for out_size in output_sizes:
            head = nn.Sequential(
                nn.LayerNorm(dim * len(category_sizes)),
                nn.Linear(dim * len(category_sizes), 64),
                nn.ReLU(),
                nn.Linear(64, out_size)
            )
            self.heads.append(head)

    def forward(self, x):
        # Embed each feature
        x_emb = [emb(x[:, i]) for i, emb in enumerate(self.embeddings)]
        x_emb = torch.stack(x_emb, dim=1)  # [batch, features, dim]

        # Transformer processing
        x_trans = self.transformer(x_emb)

        # Flatten for heads
        x_flat = x_trans.flatten(start_dim=1)

        # Multiple outputs
        return [head(x_flat) for head in self.heads]

In [26]:
class ImprovedTabTransformer(nn.Module):
    def __init__(self, category_sizes, dim, output_sizes):
        super().__init__()
        # Add feature-wise projections
        self.embeddings = nn.ModuleList([
            nn.Sequential(
                nn.Embedding(size, dim),
                nn.LayerNorm(dim)
            ) for size in category_sizes
        ])

        # Add learnable CLS token
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))

        self.transformer = nn.TransformerEncoder(
            encoder_layer=nn.TransformerEncoderLayer(
                d_model=dim,
                nhead=8,  # More attention heads
                dim_feedforward=256,
                dropout=0.2,
                batch_first=True
            ),
            num_layers=4
        )

        # Deeper output heads
        self.heads = nn.ModuleList()
        for out_size in output_sizes:
            head = nn.Sequential(
                nn.Linear(dim, dim//2),
                nn.GELU(),
                nn.LayerNorm(dim//2),
                nn.Linear(dim//2, out_size)
            )
            self.heads.append(head)

    def forward(self, x):
        x_emb = [emb(x[:, i]) for i, emb in enumerate(self.embeddings)]
        x_emb = torch.stack(x_emb, dim=1)  # [batch, features, dim]

        # Add CLS token
        cls_tokens = self.cls_token.expand(x_emb.size(0), -1, -1)
        x_emb = torch.cat((cls_tokens, x_emb), dim=1)

        x_trans = self.transformer(x_emb)

        # Use CLS token for prediction
        cls_output = x_trans[:, 0]

        return [head(cls_output) for head in self.heads]

In [27]:
# --- 5. Instantiate model ---
input_cardinality = [df[col].nunique() for col in input_cols]
output_cardinality = [df[col].nunique() for col in output_cols]

model = ImprovedTabTransformer(
    category_sizes=input_cardinality,
    dim=64,  # Increased embedding dimension
    output_sizes=output_cardinality
)

# --- 6. Training setup ---
criterions = [nn.CrossEntropyLoss() for _ in output_cols]
optimizer = optim.AdamW(model.parameters(), lr=0.001)

In [29]:
# --- 7. Training loop ---

# Add random feature masking for regularization
def augment_batch(batch_X, mask_prob=0.1):
    mask = torch.rand_like(batch_X.float()) < mask_prob
    # Replace with "unknown" token (assuming 0 is unused/unknown)
    return torch.where(mask, torch.zeros_like(batch_X), batch_X)


def train(model, X, Y, epochs=15, batch_size=32):
    model.train()
    for epoch in range(epochs):
        permutation = torch.randperm(X.size()[0])
        total_loss = 0

        for i in range(0, X.size()[0], batch_size):
            indices = permutation[i:i+batch_size]
            batch_X, batch_Y = X[indices], Y[indices]
            batch_X = augment_batch(batch_X)

            optimizer.zero_grad()
            outputs = model(batch_X)

            losses = [
                criterions[j](outputs[j], batch_Y[:, j])
                for j in range(len(output_cols))
            ]
            loss = sum(losses)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f'Epoch {epoch+1}, Loss: {total_loss / len(X)}')

# Train the model
train(model, X_train, Y_train)

Epoch 1, Loss: 0.19212554059337839
Epoch 2, Loss: 0.19173062275701008
Epoch 3, Loss: 0.1911995676841345
Epoch 4, Loss: 0.19171887934818202
Epoch 5, Loss: 0.1913279863429151
Epoch 6, Loss: 0.19082468527575808
Epoch 7, Loss: 0.19038872162229778
Epoch 8, Loss: 0.19070444953319562
Epoch 9, Loss: 0.19103745372629002
Epoch 10, Loss: 0.19008599453818675
Epoch 11, Loss: 0.1901663691029207
Epoch 12, Loss: 0.19044099221864252
Epoch 13, Loss: 0.1903779060441886
Epoch 14, Loss: 0.18995339618201143
Epoch 15, Loss: 0.1898235488344785


In [30]:
# --- 8. Evaluation ---
def evaluate(model, X, Y):
    model.eval()
    with torch.no_grad():
        outputs = model(X)
        correct = 0
        total = 0
        for j in range(len(output_cols)):
            _, predicted = torch.max(outputs[j].data, 1)
            correct += (predicted == Y[:, j]).sum().item()
            total += Y.size(0)
        accuracy = correct / total
        print(f'Accuracy: {accuracy:.4f}')

evaluate(model, X_test, Y_test)

Accuracy: 0.5562


In [31]:
def evaluate(model, X, Y):
    model.eval()
    with torch.no_grad():
        outputs = model(X)
        total_acc = 0
        feature_acc = []

        for j in range(len(output_cols)):
            _, predicted = torch.max(outputs[j].data, 1)
            correct = (predicted == Y[:, j]).sum().item()
            acc = correct / Y.size(0)
            feature_acc.append(acc)
            total_acc += acc

        avg_acc = total_acc / len(output_cols)
        print(f'Average Accuracy: {avg_acc:.4f}')
        for col, acc in zip(output_cols, feature_acc):
            print(f'{col}: {acc:.4f}')

evaluate(model, X_test, Y_test)

Average Accuracy: 0.5562
second_partner_gender: 0.8297
second_partner_age_bin: 0.6146
second_partner_school_category: 0.5169
second_partner_level_id: 0.4383
second_partner_field: 0.3816


### Sample Test - The easiest one to goo...

In [36]:
def predict_partner(sample_row, model, label_encoders):
    """
    Predict partner attributes from a sample input row

    Args:
        sample_row: List of [gender, age, school_category, level_id, field]
        model: Trained TabTransformer model
        label_encoders: Dictionary of fitted LabelEncoders

    Returns:
        Dictionary of predicted attributes
    """
    # Ensure the model is in evaluation mode
    model.eval()

    # Step 1: Prepare input with all required features
    # Now expecting all 5 features in the input
    if len(sample_row) < 5:
        raise ValueError("Input row should contain all 5 features: [gender, age, school_category, level_id, field]")

    full_sample = sample_row  # Use all provided features

    # Step 2: Encode each feature using the corresponding LabelEncoder
    input_cols = [
        'first_partner_gender',
        'first_partner_age_bin',
        'first_partner_school_category',
        'first_partner_level_id',
        'first_partner_field'
    ]

    try:
        encoded_sample = []
        for col, value in zip(input_cols, full_sample):
            # Convert to string and handle missing/unknown values
            value_str = str(value).strip() if value is not None else 'Unknown'

            # Check if the value exists in the encoder's classes
            if value_str not in label_encoders[col].classes_:
                # Find the most common class to use as default
                default_value = label_encoders[col].classes_[0]  # or use specific defaults
                print(f"Warning: Unknown value '{value_str}' for {col}, using '{default_value}' instead")
                value_str = default_value

            encoded_val = label_encoders[col].transform([value_str])[0]
            encoded_sample.append(encoded_val)

    except Exception as e:
        print(f"Encoding error: {e}")
        return None

    # Step 3: Convert to tensor and add batch dimension
    sample_tensor = torch.tensor([encoded_sample], dtype=torch.long)

    # Step 4: Get predictions
    with torch.no_grad():
        output_preds = model(sample_tensor)

    # Step 5: Decode predictions
    output_cols = [
        'second_partner_gender',
        'second_partner_age_bin',
        'second_partner_school_category',
        'second_partner_level_id',
        'second_partner_field'
    ]

    predictions = {}
    for i, col in enumerate(output_cols):
        pred_class = torch.argmax(output_preds[i], dim=1).item()
        predictions[col] = label_encoders[col].inverse_transform([pred_class])[0]

    return predictions

# Example usage with all 5 features
sample_row = [
    'Female',
    '25-29',
    'Ivy League',
    'S2',
    'Business and Financial Occupations'
]

predictions = predict_partner(sample_row, model, label_encoders)

if predictions:
    print("\n=== Predicted Partner Profile ===")
    print(f"Gender:         {predictions['second_partner_gender']}")
    print(f"Age Group:      {predictions['second_partner_age_bin']}")
    print(f"School Category: {predictions['second_partner_school_category']}")
    print(f"Education Level: {predictions['second_partner_level_id']}")
    print(f"Field of Study:  {predictions['second_partner_field']}")


=== Predicted Partner Profile ===
Gender:         Male
Age Group:      25-29
School Category: Ivy League
Education Level: S4
Field of Study:  Business and Financial Occupations
